# Computing q, k, v for ONE word

In [2]:
import numpy as np 

corpus = "the cat sat on the mat".split()
vocab = list(set(corpus))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for i, w in enumerate(vocab)}

E = np.random.randn(len(vocab), d)

In [ ]:
from scipy.special import softmax
d = 5 # dimension of my embedding for each word
dk = 3 # dimension of q/k/v - can differ from d, my choise

# The projection matrices are literally what changes the size — d is "before," dk is "after"
np.random.seed(42)
W_Q = np.random.randn(dk, d)
W_K = np.random.randn(dk, d)
W_V = np.random.randn(dk, d)

# test random x embedding to act as my input word embedding
x_1 = np.random.randn(d)

def project_qkv(x, W_Q, W_K, W_V):
    q = W_Q @ x 
    k = W_K @ x
    v = W_V @ x
    return q, k ,v

#q, k, v = project_qkv(x_1, W_Q, W_K, W_V)
#print(q, k ,v)


def project_sentence(words, E, w2i, W_Q, W_K, W_V):
    store = []
    for word in words:
        word_emb = E[w2i[word]]
        q, k, v = project_qkv(word_emb, W_Q, W_K, W_V)
        store.append([q, k, v])
    return store

store = project_sentence(corpus, E, w2i, W_Q, W_K, W_V)
#print(store)


def self_attention_scores(q_words, all_k):
    store_1 = []
    for q in q_words:
        store_2 = []
        for k in all_k:
            e_ij = np.dot(q, k)
            store_2.append(e_ij)
        store_1.append(store_2)
    return store_1 # each row represents each word q_i dot k_j score for all k in my corpus

all_q = [entry[0] for entry in store]
all_k = [entry[1] for entry in store]
all_v = [entry[2] for entry in store]

scores = self_attention_scores(all_q, all_k)
print(scores)


# lecture 5 slide 23 
def scale_scores(scores, dk):
    return np.array(scores)/np.sqrt(dk)

# helper functions used in parrallel with RNN
def attention_weights(scores):
    return softmax(scores)

def context_vector(weights, encoder_states):
    context = []
    for w, h in zip(weights, encoder_states):
        context.append(w * h)
    return sum(context)

# v = the value vector for one word (from project_qkv's third output)
def self_attention_output(scores, all_v, dk): 
    output = []
    for word_score in scores:
        scaled_score = scale_scores(word_score, dk)
        context_score = context_vector(attention_weights(scaled_score), all_v)
        output.append(context_score)
    return output

attention_output = self_attention_output(scores, all_v, dk)
print(attention_output)

[[np.float64(3.9960142545700372), np.float64(0.5367543712745453), np.float64(2.027577825556401), np.float64(-0.151942157722675), np.float64(3.9960142545700372), np.float64(-2.292987003717757)], [np.float64(6.354628643434886), np.float64(-1.8808485991212922), np.float64(10.151610885995876), np.float64(-6.13368839350705), np.float64(6.354628643434886), np.float64(-8.344849205933983)], [np.float64(6.974535237274961), np.float64(-0.2116348400164485), np.float64(1.685200841263474), np.float64(-0.34435967802958634), np.float64(6.974535237274961), np.float64(-3.2207938652956924)], [np.float64(-1.3657075958019793), np.float64(-1.9688097004053025), np.float64(-5.353388981975176), np.float64(0.8234633727980443), np.float64(-1.3657075958019793), np.float64(3.027006627681293)], [np.float64(3.9960142545700372), np.float64(0.5367543712745453), np.float64(2.027577825556401), np.float64(-0.151942157722675), np.float64(3.9960142545700372), np.float64(-2.292987003717757)], [np.float64(1.2781485299289161

# Feed-forwrad network

In [ ]:
hidden_size = 4
W1 = np.random.randn(hidden_size, dk)
b1 = np.random.randn(hidden_size)
W2 = np.random.randn(dk, hidden_size)
b2 = np.random.randn(dk)

def relu(x):
    return np.maximum(x, 0)

def feed_forward(x, W1, b1, W2, b2):
    # x = one word's attention output
    inner = W1 @ x +b1
    layer1 = relu(inner)
    output = W2 @ layer1 + b2
    return output

test = feed_forward(attention_output[0], W1, b1, W2, b2)
print(test)

def residual(input_x, output_x):
    return input_x + output_x

def layer_norm(x, epsilon = 1e-5):
    # normalizing for just one word 
    u = np.mean(x)
    sigma = np.std(x)
    return (x - u )/(sigma + epsilon)



[ 1.14141013 -2.09155407  0.03298242]
